# Interactive RGB Rainbow LED Effects
## Accordion Hardware Visualization with Threading Control

This notebook demonstrates real-time RGB rainbow effects on a 2D LED array using the Accordion.py hardware wrapper with thread-based control and event-driven stopping mechanisms.

## Section 1: Import Libraries and Initialize Hardware

In [2]:
# Import standard library modules
import threading
import time
import colorsys
import sys
import os
from typing import List, Optional, Tuple

# Add the current directory to path to import accordion
sys.path.insert(0, os.getcwd())

# Import the accordion wrapper
import accordion as acc

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# Initialize hardware connection
print("Connecting to Accordion hardware...")
try:
    acc.attach("py", "agentdemo.local")
    device_id = acc.get_identification()
    print(f"✓ Device connected: {device_id}")
except Exception as e:
    print(f"✗ Connection failed: {e}")

Connecting to Accordion hardware...
✓ Device connected: agentdemo
Name=Control Module 32ch, ID=ESH10000023, S/N=A002274, Rev=4
Name=AGENT Q2 Base, ID=ESH10000158, S/N=A002814, Rev=4
Name=I2Cx4 Long Range Module, ID=ESH10000239, S/N=A000000, Rev=2
Name=MPIO-96 SPI Module, ID=ESH10000568, S/N=A000000, Rev=0
Name=AGENT Q2 TOP, ID=ESH10000183, S/N=A002838, Rev=6
Name=N-Module 6xIDC, ID=ESH10000355, S/N=A002612, Rev=6
Name=N-Module 6xIDC, ID=ESH10000355, S/N=A002613, Rev=6
Name=N-Module 6xIDC, ID=ESH10000355, S/N=A002611, Rev=6
Name=N-Module 6xIDC, ID=ESH10000355, S/N=A002614, Rev=6
Name=N5 TOP, ID=ESH10000359, S/N=A002911, Rev=2
Name=PSU1, ID=ESH10000533, S/N=A002727, Rev=3
Name=Active load 2ch, ID=ESH10000536, S/N=A002848, Rev=2



## Section 2: Color Utility Functions

In [4]:
def hsv_to_hex(h: float, s: float = 1.0, v: float = 1.0) -> str:
    """
    Convert HSV color to hexadecimal RGB string.
    
    Args:
        h: Hue (0-1)
        s: Saturation (0-1)
        v: Value/Brightness (0-1)
    
    Returns:
        Hex color string (e.g., '#FF5733')
    """
    r, g, b = colorsys.hsv_to_rgb(h % 1.0, s, v)
    r_int = int(max(0, min(255, r * 255)))
    g_int = int(max(0, min(255, g * 255)))
    b_int = int(max(0, min(255, b * 255)))
    return f"#{r_int:02x}{g_int:02x}{b_int:02x}"

def generate_rainbow_hues(count: int) -> List[str]:
    """Generate `count` rainbow color hex values evenly distributed across the hue spectrum."""
    return [hsv_to_hex(i / count) for i in range(count)]

# Test the color functions
print("Color function test:")
test_colors = generate_rainbow_hues(5)
print(f"5-color rainbow: {test_colors}")

Color function test:
5-color rainbow: ['#ff0000', '#cbff00', '#00ff66', '#0066ff', '#cc00ff']


## Section 3: Discover and Map LED Channels

In [5]:
# Discover LED channels
all_channels = acc.get_channel_names()
led_channels = [ch for ch in sorted(all_channels) if "ESH10000355" in ch and not "LUMINANCE" in ch]

print(f"Found {len(led_channels)} LED channels:")
for i, ch in enumerate(led_channels):
    print(f"  [{i}] {ch}")

# Define 2D array dimensions
ROWS, COLS = 6, 4  # Adjust based on your actual LED layout
EXPECTED_LEDS = ROWS * COLS

print(f"\nConfiguring for {ROWS}x{COLS} = {EXPECTED_LEDS} LEDs")
if len(led_channels) >= EXPECTED_LEDS:
    led_array_channels = led_channels[:EXPECTED_LEDS]
    print(f"✓ Using first {EXPECTED_LEDS} channels")
else:
    led_array_channels = led_channels
    print(f"⚠ Only {len(led_channels)} channels available")

Found 24 LED channels:
  [0] 0.11.ESH10000355.A1
  [1] 0.11.ESH10000355.B1
  [2] 0.11.ESH10000355.C1
  [3] 0.11.ESH10000355.D1
  [4] 0.11.ESH10000355.E1
  [5] 0.11.ESH10000355.F1
  [6] 0.12.ESH10000355.A2
  [7] 0.12.ESH10000355.B2
  [8] 0.12.ESH10000355.C2
  [9] 0.12.ESH10000355.D2
  [10] 0.12.ESH10000355.E2
  [11] 0.12.ESH10000355.F2
  [12] 0.13.ESH10000355.A3
  [13] 0.13.ESH10000355.B3
  [14] 0.13.ESH10000355.C3
  [15] 0.13.ESH10000355.D3
  [16] 0.13.ESH10000355.E3
  [17] 0.13.ESH10000355.F3
  [18] 0.14.ESH10000355.A4
  [19] 0.14.ESH10000355.B4
  [20] 0.14.ESH10000355.C4
  [21] 0.14.ESH10000355.D4
  [22] 0.14.ESH10000355.E4
  [23] 0.14.ESH10000355.F4

Configuring for 6x4 = 24 LEDs
✓ Using first 24 channels


## Section 4: 2D Array Helper Functions

In [6]:
def coords_to_index(row: int, col: int, cols: int) -> int:
    """Convert 2D (row, col) coordinates to linear index."""
    return row * cols + col

def index_to_coords(index: int, cols: int) -> Tuple[int, int]:
    """Convert linear index to 2D (row, col) coordinates."""
    row = index // cols
    col = index % cols
    return row, col

def set_led_at(row: int, col: int, color: str, channels: List[str], cols: int) -> None:
    """Set LED color at 2D position."""
    idx = coords_to_index(row, col, cols)
    if 0 <= idx < len(channels):
        acc.set_value(channels[idx], color)

def set_all_leds(color: str, channels: List[str]) -> None:
    """Set all LEDs to the same color."""
    for channel in channels:
        acc.set_value(channel, color)

def clear_leds(channels: List[str]) -> None:
    """Turn off all LEDs."""
    set_all_leds('black', channels)

print("LED helper functions defined")

LED helper functions defined


## Section 5: Rainbow Animation Generator

In [7]:
def rainbow_wave_frame(frame: int, speed: float, total_leds: int, 
                       channels: List[str], cols: int) -> None:
    """
    Generate a rainbow wave effect.
    
    Args:
        frame: Current frame number
        speed: Animation speed multiplier
        total_leds: Total number of LEDs
        channels: List of LED channel names
        cols: Number of columns for 2D indexing
    """
    for idx in range(total_leds):
        # Calculate hue based on position and animation frame
        hue = ((idx + frame * speed) % total_leds) / total_leds
        color = hsv_to_hex(hue, s=1.0, v=1.0)
        
        # Update the LED
        if idx < len(channels):
            acc.set_value(channels[idx], color)

def rainbow_diagonal_frame(frame: int, speed: float, rows: int, cols: int,
                          channels: List[str]) -> None:
    """
    Generate a diagonal rainbow effect that travels across the array.
    """
    for row in range(rows):
        for col in range(cols):
            # Hue based on diagonal position + animation offset
            hue = ((row + col + frame * speed) % (rows + cols)) / (rows + cols)
            color = hsv_to_hex(hue)
            set_led_at(row, col, color, channels, cols)

def pulse_frame(frame: int, base_hue: float, speed: float, channels: List[str]) -> None:
    """
    Generate a pulsing effect - all LEDs brighten and dim together.
    """
    import math
    # Sinusoidal brightness variation
    brightness = 0.5 + 0.5 * math.sin(frame * speed * 0.1)
    color = hsv_to_hex(base_hue, s=1.0, v=brightness)
    set_all_leds(color, channels)

print("Animation generators defined")

Animation generators defined


## Section 6: Threading with Event-Based Control

In [8]:
class LEDAnimationControl:
    """Thread-safe controller for LED animations with event-based stopping."""
    
    def __init__(self, channels: List[str], rows: int, cols: int):
        self.channels = channels
        self.rows = rows
        self.cols = cols
        self.total_leds = rows * cols
        
        self._thread: Optional[threading.Thread] = None
        self._stop_event = threading.Event()
        self._is_running = False
    
    def _run_animation(self, animation_func, update_interval: float = 0.05, **kwargs) -> None:
        """Internal method to run animation in a thread."""
        frame = 0
        self._stop_event.clear()
        self._is_running = True
        
        try:
            while not self._stop_event.is_set():
                try:
                    animation_func(frame, **kwargs)
                    time.sleep(update_interval)
                    frame += 1
                except Exception as e:
                    print(f"Error in animation: {e}")
                    break
        finally:
            self._is_running = False
            clear_leds(self.channels)
    
    def start_animation(self, animation_func, update_interval: float = 0.05, **kwargs) -> None:
        """
        Start an LED animation in a background thread.
        
        Args:
            animation_func: Function that takes (frame, **kwargs) and updates LEDs
            update_interval: Time between frames in seconds
            **kwargs: Additional arguments passed to animation_func
        """
        if self._thread is not None and self._thread.is_alive():
            self.stop_animation()
            self._thread.join(timeout=1.0)
        
        self._thread = threading.Thread(
            target=self._run_animation,
            args=(animation_func, update_interval),
            kwargs=kwargs,
            daemon=True
        )
        self._thread.start()
    
    def stop_animation(self) -> None:
        """Stop the currently running animation."""
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
    
    def is_running(self) -> bool:
        """Check if animation is currently running."""
        return self._is_running
    
    def wait_until_stopped(self, timeout: Optional[float] = None) -> bool:
        """Wait for animation to stop. Returns True if stopped, False if timeout."""
        if self._thread is None:
            return True
        self._thread.join(timeout=timeout)
        return not self._thread.is_alive()

# Create controller instance
anim_control = LEDAnimationControl(led_array_channels, ROWS, COLS)
print("Animation controller created")

Animation controller created


## Section 7: Run Rainbow Animation

Start a rainbow wave effect on your LED array. The animation runs in a background thread, allowing the notebook to remain responsive.

In [9]:
# Start rainbow wave animation
print("Starting rainbow wave animation...")
print(f"Animating {len(led_array_channels)} LEDs in a {ROWS}x{COLS} array")

anim_control.start_animation(
    animation_func=rainbow_wave_frame,
    update_interval=0.05,
    speed=1.0,
    total_leds=len(led_array_channels),
    channels=led_array_channels,
    cols=COLS
)

print(f"✓ Animation running: {anim_control.is_running()}")
print("\nUse the 'Stop Animation' cell below to stop it at any time.")

Starting rainbow wave animation...
Animating 24 LEDs in a 6x4 array
✓ Animation running: True

Use the 'Stop Animation' cell below to stop it at any time.


## Section 8: Control Animation

In [ ]:
# Stop animation
anim_control.stop_animation()
print("✓ Animation stopped")
print(f"  Running: {anim_control.is_running()}")

In [ ]:
# Try other animation effects
# Rainbow diagonal
print("Starting rainbow diagonal animation...")
anim_control.start_animation(
    animation_func=rainbow_diagonal_frame,
    update_interval=0.05,
    speed=0.5,
    rows=ROWS,
    cols=COLS,
    channels=led_array_channels
)

print(f"✓ Animation running: {anim_control.is_running()}")

In [ ]:
# Try pulsing effect
print("Starting pulse animation (cyan color)...")
anim_control.stop_animation()  # Stop current animation

anim_control.start_animation(
    animation_func=pulse_frame,
    update_interval=0.05,
    base_hue=0.5,  # Cyan
    speed=1.0,
    channels=led_array_channels
)

print(f"✓ Pulse animation running: {anim_control.is_running()}")

## Create Your Own Custom Animation

Define a custom animation function and run it:


In [ ]:
def custom_checkerboard(frame: int, speed: float, rows: int, cols: int, 
                       channels: List[str]) -> None:
    """
    Checkerboard pattern that alternates colors.
    """
    color1 = hsv_to_hex((frame * speed) % 1.0)
    color2 = hsv_to_hex(((frame * speed) + 0.5) % 1.0)
    
    for row in range(rows):
        for col in range(cols):
            # Checkerboard pattern
            is_alternate = (row + col) % 2 == 0
            color = color1 if is_alternate else color2
            set_led_at(row, col, color, channels, cols)

# Run custom animation
print("Starting custom checkerboard animation...")
anim_control.stop_animation()

anim_control.start_animation(
    animation_func=custom_checkerboard,
    update_interval=0.05,
    speed=0.3,
    rows=ROWS,
    cols=COLS,
    channels=led_array_channels
)

print(f"✓ Custom animation running: {anim_control.is_running()}")

## Cleanup

In [ ]:
# Stop animation and disconnect hardware
print("Cleaning up...")
anim_control.stop_animation()
clear_leds(led_array_channels)

# Disconnect from hardware
acc.detach()
print("✓ Hardware disconnected")
print("Done!")

## Section 2.5: LED Position Mapping

The hardware features 4 modules arranged in a 2x2 pattern, with each module containing 6 LEDs in a 2x3 layout. This mapping system allows you to reference LEDs by their physical position or by module + position within module.

In [ ]:
# Import the mapping class from the example module
import sys
import os
sys.path.insert(0, os.getcwd())

# Import LEDMapping
exec(open('example.py').read().split('if __name__')[0])  # Load classes only

# Create the LED mapping for 2x2 modules (2x3 LEDs per module)
mapping = LEDMapping(led_array_channels)
mapping.print_mapping_info()

# Visualize the mapping
print("\nPhysical LED Grid Layout:")
print("=" * 50)
for row in range(mapping.total_rows):
    row_str = f"Row {row}: "
    for col in range(mapping.total_cols):
        try:
            idx = mapping.position_to_index(row, col)
            row_str += f"[{idx:2d}] "
        except:
            row_str += "[-] "
    print(row_str)
print("=" * 50)

print("\nModule Arrangement:")
print("  Module(0,0) | Module(0,1)")
print("  -----------+-----------")
print("  Module(1,0) | Module(1,1)")

## Section 6.5: Initialize LEDArray2D with Mapping

Create an LEDArray2D controller that uses the physical position mapping.

In [1]:
# Create LED array controller with mapping
led_controller = LEDArray2D(led_array_channels, mapping)

print(f"✓ LED Controller initialized")
print(f"  Total LEDs: {led_controller.total_leds}")
print(f"  Grid dimensions: {led_controller.rows} rows x {led_controller.cols} columns")
print(f"  Modules: {mapping.num_modules_rows}x{mapping.num_modules_cols} ({mapping.leds_per_module_rows}x{mapping.leds_per_module_cols} LEDs per module)")

NameError: name 'LEDArray2D' is not defined